# 实验4.3 昇腾香橙派基于 Ascend C 的基础算子开发实验

> **实验名称**：基于 Ascend C 的基础算子开发实验（昇腾香橙派 310B3）
> **目标硬件**：香橙派开发板（昇腾 310B3 NPU）
> **软件环境**：CANN Toolkit · torch_npu · Python 3
> **建议学时**：4 学时

## 实验导学

本实验基于已预先调通的**香橙派昇腾 310B3 开发板**环境，聚焦 Ascend C 算子开发中的关键环节。学生将在已配置完成的环境中，通过 **Python + torch_npu** 完成**四个递进式实验**的代码研读、上板运行与结果验证。

> **注意**：本 Notebook 主要以**香橙派开发板（昇腾 310B3）**为主。Notebook 中的代码块仅用于**教学说明**，无需在当前环境仿真运行；在香橙派上的实际操作请参照实验手册中的 SSH 操作步骤，运行 `code/` 目录下的 Python 脚本。

---

## 一、实验概述

### 1.1 实验背景

算子是 AI 模型中最基本的计算单元。当我们使用 PyTorch 书写 `z = x + y` 时，框架底层实际调用的是经过深度优化的加法算子。**算子的执行效率直接决定了模型在芯片上的整体性能**。

Ascend C 是华为面向昇腾 NPU 推出的算子开发编程语言，在 C/C++ 基础上扩展了一套针对 AI 计算与数据搬运的编程接口。本实验通过 Python + torch_npu 在香橙派 310B3 上复现 Ascend C 算子开发中的四个递进式案例，帮助学生理解算子开发的核心概念。

### 1.2 四个递进式实验

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">阶段</th>
<th style="text-align: left;">实验内容</th>
<th style="text-align: left;">教学重点</th>
<th style="text-align: left;">核心变化</th>
</tr>
<tr>
<td style="text-align: left;"><strong>阶段一</strong></td>
<td style="text-align: left;">双向量加法（8核）</td>
<td style="text-align: left;">掌握算子开发基本流程</td>
<td style="text-align: left;">基准实现</td>
</tr>
<tr>
<td style="text-align: left;"><strong>阶段二</strong></td>
<td style="text-align: left;">双向量加法（32核）</td>
<td style="text-align: left;">理解多核并行与数据切分</td>
<td style="text-align: left;">核数 8→32</td>
</tr>
<tr>
<td style="text-align: left;"><strong>阶段三</strong></td>
<td style="text-align: left;">三向量加法</td>
<td style="text-align: left;">掌握多输入算子的实现</td>
<td style="text-align: left;">输入 2→3</td>
</tr>
<tr>
<td style="text-align: left;"><strong>阶段四</strong></td>
<td style="text-align: left;">双向量乘法</td>
<td style="text-align: left;">理解不同计算指令的使用</td>
<td style="text-align: left;">Add→Mul</td>
</tr>
</table>

### 1.3 实验目标

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">目标类型</th>
<th style="text-align: left;">内容</th>
</tr>
<tr>
<td style="text-align: left;"><strong>知识目标</strong></td>
<td style="text-align: left;">理解 GM/LM 分工、多核并行与数据切分、双缓冲、三级流水线</td>
</tr>
<tr>
<td style="text-align: left;"><strong>能力目标</strong></td>
<td style="text-align: left;">在香橙派 310B3 上完成四阶段算子的代码研读、运行与验证</td>
</tr>
<tr>
<td style="text-align: left;"><strong>素养目标</strong></td>
<td style="text-align: left;">形成"实践→总结→贡献"的闭环意识</td>
</tr>
</table>

### 1.4 实验环境

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">配置项</th>
<th style="text-align: left;">值</th>
</tr>
<tr>
<td style="text-align: left;">开发板</td>
<td style="text-align: left;">香橙派 AIPro（昇腾 310B3）</td>
</tr>
<tr>
<td style="text-align: left;">NPU 芯片</td>
<td style="text-align: left;">昇腾 310B3</td>
</tr>
<tr>
<td style="text-align: left;">主机 IP</td>
<td style="text-align: left;">192.168.137.101</td>
</tr>
<tr>
<td style="text-align: left;">开发板 IP</td>
<td style="text-align: left;">192.168.137.100</td>
</tr>
<tr>
<td style="text-align: left;">登录方式</td>
<td style="text-align: left;">SSH（MobaXterm）</td>
</tr>
<tr>
<td style="text-align: left;">用户名</td>
<td style="text-align: left;">root</td>
</tr>
<tr>
<td style="text-align: left;">Python 框架</td>
<td style="text-align: left;">torch + torch_npu</td>
</tr>
<tr>
<td style="text-align: left;">供电接口</td>
<td style="text-align: left;">右侧 Type-C（左侧为调试口）</td>
</tr>
</table>

---

## 二、实验原理

### 2.1 算子与 Ascend C 编程语言

向量加法算子的数学表达式为 `z = x + y`。要在 NPU 上高效执行，必须回答三个问题：

1. **数据放在哪里** → 全局内存（GM）vs 片上内存（LM）
2. **由谁计算** → AI Core 的 Vector 计算单元
3. **如何组织计算** → 三级流水线 + 多核并行

### 2.2 存储体系

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">存储层级</th>
<th style="text-align: left;">位置</th>
<th style="text-align: left;">特点</th>
</tr>
<tr>
<td style="text-align: left;">全局内存（GM）</td>
<td style="text-align: left;">AI Core 外部</td>
<td style="text-align: left;">容量大、访问慢，类比 DDR</td>
</tr>
<tr>
<td style="text-align: left;">片上内存（LM）</td>
<td style="text-align: left;">AI Core 内部</td>
<td style="text-align: left;">容量小、速度极快</td>
</tr>
</table>

> Ascend C 的矢量计算接口操作数必须是 LM 中的 LocalTensor，因此固定套路是：**GM 搬入 LM → LM 计算 → 结果搬回 GM**。在 Python + torch_npu 中，这一过程对应 `x.to(device)` 将数据从 CPU（Host）搬到 NPU（Device），NPU 上计算后再 `.cpu()` 搬回。

### 2.3 三级流水线（CopyIn → Compute → CopyOut）

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">阶段</th>
<th style="text-align: left;">任务</th>
<th style="text-align: left;">Python/torch_npu 对应</th>
</tr>
<tr>
<td style="text-align: left;"><strong>CopyIn</strong></td>
<td style="text-align: left;">GM → LM 数据搬运</td>
<td style="text-align: left;"><code>x.to(device)</code> 将数据从 Host 搬到 NPU</td>
</tr>
<tr>
<td style="text-align: left;"><strong>Compute</strong></td>
<td style="text-align: left;">LM 上执行计算</td>
<td style="text-align: left;"><code>x_npu + y_npu</code> 在 NPU 上执行向量运算</td>
</tr>
<tr>
<td style="text-align: left;"><strong>CopyOut</strong></td>
<td style="text-align: left;">LM → GM 结果搬回</td>
<td style="text-align: left;"><code>z.cpu()</code> 将结果从 NPU 搬回 Host</td>
</tr>
</table>

### 2.4 多核并行与数据切分

以阶段一为例，将 `8 × 2048 = 16384` 个元素平均分配给 **8 个 AI Core** 并行处理：

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">常量</th>
<th style="text-align: left;">取值</th>
<th style="text-align: left;">含义</th>
</tr>
<tr>
<td style="text-align: left;"><code>total_length</code></td>
<td style="text-align: left;">16384</td>
<td style="text-align: left;">数据总长度</td>
</tr>
<tr>
<td style="text-align: left;"><code>use_core_num</code></td>
<td style="text-align: left;">8</td>
<td style="text-align: left;">使用的核数</td>
</tr>
<tr>
<td style="text-align: left;"><code>block_length</code></td>
<td style="text-align: left;">2048</td>
<td style="text-align: left;">每核处理长度</td>
</tr>
</table>

在 Python 代码中，多核并行由 torch_npu 底层自动调度，用户只需将数据搬到 NPU 设备并执行向量运算，框架会自动将数据切分到多个 AI Core 上并行处理。

### 2.5 双缓冲机制

`BUFFER_NUM = 2` 使 CopyIn 和 Compute 并行执行：当前 Tile 计算时，下一 Tile 数据已在搬运中，**隐藏访存延迟**。在 torch_npu 底层，这一机制由 NPU 驱动自动管理。


---

## 三、Python 代码文件结构

在香橙派上，本实验的 Python 代码位于 `code/` 目录：

```text
code/
├── add_8core_orangepi.py     # 阶段一：双向量加法 8核 (z = x + y)
├── add_32core_orangepi.py    # 阶段二：双向量加法 32核 (z = x + y)
├── add3_orangepi.py          # 阶段三：三向量加法 (z = x + y + w)
├── mul_orangepi.py           # 阶段四：双向量乘法 (z = x * y)
└── run_orangepi.sh           # 一键运行脚本
```

**四阶段输入输出对照表**：

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">阶段</th>
<th style="text-align: left;">文件</th>
<th style="text-align: left;">输入</th>
<th style="text-align: left;">算子</th>
<th style="text-align: left;">预期输出</th>
<th style="text-align: left;">核数</th>
</tr>
<tr>
<td style="text-align: left;">阶段一</td>
<td style="text-align: left;"><code>add_8core_orangepi.py</code></td>
<td style="text-align: left;">x=1.2, y=2.3</td>
<td style="text-align: left;">z=x+y</td>
<td style="text-align: left;">3.5</td>
<td style="text-align: left;">8</td>
</tr>
<tr>
<td style="text-align: left;">阶段二</td>
<td style="text-align: left;"><code>add_32core_orangepi.py</code></td>
<td style="text-align: left;">x=2.2, y=2.3</td>
<td style="text-align: left;">z=x+y</td>
<td style="text-align: left;">4.5</td>
<td style="text-align: left;">32</td>
</tr>
<tr>
<td style="text-align: left;">阶段三</td>
<td style="text-align: left;"><code>add3_orangepi.py</code></td>
<td style="text-align: left;">x=1.2, y=2.3, w=3.4</td>
<td style="text-align: left;">z=x+y+w</td>
<td style="text-align: left;">6.9</td>
<td style="text-align: left;">8</td>
</tr>
<tr>
<td style="text-align: left;">阶段四</td>
<td style="text-align: left;"><code>mul_orangepi.py</code></td>
<td style="text-align: left;">x=1.2, y=2.3</td>
<td style="text-align: left;">z=x*y</td>
<td style="text-align: left;">2.76</td>
<td style="text-align: left;">8</td>
</tr>
</table>

---

## 四、阶段一：双向量加法（8核）—— 基准实现

### 4.1 算子分析

- 输入 shape 固定为 (8, 2048)，共 16384 个元素
- 输入类型为 float
- 固定使用 8 个核
- 输入 x 全部填充 1.2，y 全部填充 2.3，预期输出 z 全部为 3.5

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">项目</th>
<th style="text-align: left;">规格</th>
</tr>
<tr>
<td style="text-align: left;">算子类型</td>
<td style="text-align: left;">Add</td>
</tr>
<tr>
<td style="text-align: left;">输入 x</td>
<td style="text-align: left;">shape (8,2048), float</td>
</tr>
<tr>
<td style="text-align: left;">输入 y</td>
<td style="text-align: left;">shape (8,2048), float</td>
</tr>
<tr>
<td style="text-align: left;">输出 z</td>
<td style="text-align: left;">shape (8,2048), float</td>
</tr>
<tr>
<td style="text-align: left;">使用核数</td>
<td style="text-align: left;">8</td>
</tr>
</table>

### 4.2 代码文件：`code/add_8core_orangepi.py`

#### 程序块 1：导入与验证函数

```python
import torch
import torch_npu


def verify_result(output, golden):
    """精度验证：逐元素比对输出与 golden 值"""
    print(f"Output: {' '.join(f'{v:.1f}' for v in output[:20].tolist())}...")
    print(f"Golden: {' '.join(f'{v:.1f}' for v in golden[:20].tolist())}...")
    if torch.allclose(output, golden, rtol=1e-6, atol=1e-6):
        print("[Success] 精度验证通过！")
        return 0
    else:
        print("[Failed] 精度验证失败！")
        return 1
```

**说明**：
- `torch_npu` 是昇腾 NPU 的 PyTorch 后端插件，安装后 `torch.device('npu:0')` 即可使用昇腾 310B3 的 AI Core 进行计算。
- `verify_result` 函数对应 Ascend C 中的 `VerifyResult`，打印前 20 个元素并用 `torch.allclose` 做浮点精度比对。

#### 程序块 2：数据参数定义

```python
use_core_num = 8
block_length = 2048
total_length = use_core_num * block_length  # 16384
value_x = 1.2
value_y = 2.3
```

**说明**：对应 Ascend C 算子中的常量定义。`total_length = 8 × 2048 = 16384`，平均分配到 8 个核，每核处理 2048 个元素。`value_x = 1.2` 和 `value_y = 2.3` 是输入向量的填充值。

#### 程序块 3：NPU 设备初始化

```python
device = torch.device('npu:0')
```

**说明**：对应 Ascend C 中的 `aclInit(nullptr)` + `aclrtSetDevice(deviceId)`。`npu:0` 表示使用第 0 块 NPU 设备（香橙派 310B3 只有一块 NPU）。

#### 程序块 4：输入数据生成（Host 侧）

```python
x = torch.full((total_length,), value_x, dtype=torch.float32)
y = torch.full((total_length,), value_y, dtype=torch.float32)
```

**说明**：对应 Ascend C `main` 函数中的 `std::vector<float> x(totalLength, valueX)`。在 CPU（Host）上生成 16384 个 1.2 和 16384 个 2.3 的向量。

#### 程序块 5：数据搬移到 NPU（Host → Device）

```python
x_npu = x.to(device)
y_npu = y.to(device)
```

**说明**：对应 Ascend C 中的 `aclrtMemcpy(xDevice, ..., xHost, ..., ACL_MEMCPY_HOST_TO_DEVICE)`。将数据从 Host 内存搬运到 NPU 的 Global Memory（GM）。

#### 程序块 6：NPU 上执行向量加法

```python
z_npu = x_npu + y_npu
```

**说明**：这是算子的核心计算步骤，对应 Ascend C 核函数中的 `Add(zLocal, xLocal, yLocal, tileLength)`。底层由 8 个 AI Core 并行处理，每个核处理 2048 个元素，核内再通过三级流水线（CopyIn → Compute → CopyOut）和双缓冲机制高效执行。

#### 程序块 7：结果搬回 Host 与精度验证

```python
z = z_npu.cpu()
golden = torch.full((total_length,), value_x + value_y, dtype=torch.float32)
return verify_result(z, golden)
```

**说明**：`.cpu()` 对应 `aclrtMemcpy(zHost, ..., zDevice, ..., ACL_MEMCPY_DEVICE_TO_HOST)`，将 NPU 计算结果搬回 Host。golden 值为 `1.2 + 2.3 = 3.5`，验证输出是否全部为 3.5。

### 4.3 运行预期结果

```text
计算设备: npu:0
Output: 3.5 3.5 3.5 3.5 3.5 3.5 3.5 3.5 3.5 3.5 3.5 3.5 3.5 3.5 3.5 3.5 3.5 3.5 3.5 3.5...
Golden: 3.5 3.5 3.5 3.5 3.5 3.5 3.5 3.5 3.5 3.5 3.5 3.5 3.5 3.5 3.5 3.5 3.5 3.5 3.5 3.5...
[Success] 精度验证通过！
```

**香橙派实际运行结果截图**：

![阶段一运行结果](images/image1.png)

> **输入→输出对应关系**：输入 x = [1.2, 1.2, ..., 1.2]（16384 个 1.2），y = [2.3, 2.3, ..., 2.3]（16384 个 2.3），算子执行加法 z[i] = x[i] + y[i] = 1.2 + 2.3 = 3.5，输出 z = [3.5, 3.5, ..., 3.5]（16384 个 3.5）。

---

## 五、阶段二：双向量加法（32核）—— 多核并行扩展

### 5.1 与阶段一的区别

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">项目</th>
<th style="text-align: left;">阶段一</th>
<th style="text-align: left;">阶段二</th>
</tr>
<tr>
<td style="text-align: left;">输入 shape</td>
<td style="text-align: left;">(8, 2048)</td>
<td style="text-align: left;">(32, 2048)</td>
</tr>
<tr>
<td style="text-align: left;">数据总量</td>
<td style="text-align: left;">16,384 元素</td>
<td style="text-align: left;">65,536 元素</td>
</tr>
<tr>
<td style="text-align: left;">使用核数</td>
<td style="text-align: left;">8</td>
<td style="text-align: left;">32</td>
</tr>
<tr>
<td style="text-align: left;">每核处理</td>
<td style="text-align: left;">2,048 元素</td>
<td style="text-align: left;">2,048 元素</td>
</tr>
<tr>
<td style="text-align: left;">value_x</td>
<td style="text-align: left;">1.2</td>
<td style="text-align: left;">2.2</td>
</tr>
<tr>
<td style="text-align: left;">预期输出</td>
<td style="text-align: left;">3.5</td>
<td style="text-align: left;">4.5</td>
</tr>
</table>

### 5.2 代码文件：`code/add_32core_orangepi.py`

#### 程序块 1：数据参数定义（关键变化）

```python
use_core_num = 32              # 8 -> 32 核并行
block_length = 2048
total_length = use_core_num * block_length  # 65536
value_x = 2.2                  # 与阶段一的 1.2 区分
value_y = 2.3
```

**说明**：与阶段一相比，仅修改三处：`use_core_num` 从 8 改为 32，`total_length` 从 16384 改为 65536，`value_x` 从 1.2 改为 2.2。每核处理数据量 `block_length` 保持 2048 不变。

#### 程序块 2：NPU 计算与验证

```python
device = torch.device('npu:0')
x = torch.full((total_length,), value_x, dtype=torch.float32)
y = torch.full((total_length,), value_y, dtype=torch.float32)
x_npu = x.to(device)
y_npu = y.to(device)
z_npu = x_npu + y_npu       # 32 个 AI Core 并行处理
z = z_npu.cpu()
golden = torch.full((total_length,), value_x + value_y, dtype=torch.float32)
return verify_result(z, golden)
```

**说明**：计算逻辑与阶段一完全相同，仅数据量和核数不同。32 个核并行处理 65536 个元素，每个核仍处理 2048 个元素。golden 值为 `2.2 + 2.3 = 4.5`。

### 5.3 运行预期结果

```text
计算设备: npu:0
Output: 4.5 4.5 4.5 4.5 4.5 4.5 4.5 4.5 4.5 4.5 4.5 4.5 4.5 4.5 4.5 4.5 4.5 4.5 4.5 4.5...
Golden: 4.5 4.5 4.5 4.5 4.5 4.5 4.5 4.5 4.5 4.5 4.5 4.5 4.5 4.5 4.5 4.5 4.5 4.5 4.5 4.5...
[Success] 精度验证通过！
```

**香橙派实际运行结果截图**：

![阶段二运行结果](images/image2.png)

> **与阶段一对比**：阶段一输出全为 3.5（1.2+2.3），阶段二输出全为 4.5（2.2+2.3）。输入常量不同导致输出不同，但算子逻辑（加法）相同。32 个核并行处理，验证了核数扩展时数据切分的正确性。

---

## 六、阶段三：三向量加法 —— 多输入算子扩展

### 6.1 与阶段一的区别

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">项目</th>
<th style="text-align: left;">阶段一</th>
<th style="text-align: left;">阶段三</th>
</tr>
<tr>
<td style="text-align: left;">输入数量</td>
<td style="text-align: left;">2 (x, y)</td>
<td style="text-align: left;">3 (x, y, w)</td>
</tr>
<tr>
<td style="text-align: left;">计算公式</td>
<td style="text-align: left;">z = x + y</td>
<td style="text-align: left;">z = x + y + w</td>
</tr>
<tr>
<td style="text-align: left;">value_w</td>
<td style="text-align: left;">无</td>
<td style="text-align: left;">3.4</td>
</tr>
<tr>
<td style="text-align: left;">预期输出</td>
<td style="text-align: left;">3.5</td>
<td style="text-align: left;">6.9</td>
</tr>
</table>

### 6.2 代码文件：`code/add3_orangepi.py`

#### 程序块 1：数据参数定义（增加第三个输入）

```python
use_core_num = 8
block_length = 2048
total_length = use_core_num * block_length  # 16384
value_x = 1.2
value_y = 2.3
value_w = 3.4                  # 新增第三个输入的填充值
```

**说明**：与阶段一相比，增加 `value_w = 3.4`，对应第三个输入向量 w。

#### 程序块 2：输入数据生成（增加 w）

```python
x = torch.full((total_length,), value_x, dtype=torch.float32)
y = torch.full((total_length,), value_y, dtype=torch.float32)
w = torch.full((total_length,), value_w, dtype=torch.float32)  # 新增
```

**说明**：在阶段一基础上增加第三个输入向量 w，全部填充 3.4。

#### 程序块 3：数据搬移到 NPU（增加 w）

```python
x_npu = x.to(device)
y_npu = y.to(device)
w_npu = w.to(device)            # 新增
```

#### 程序块 4：NPU 上执行三向量加法（核心变化）

```python
z_npu = x_npu + y_npu + w_npu
```

**说明**：对应 Ascend C Compute 函数中的两次 Add 指令：
- 第 1 次：`Add(zLocal, xLocal, yLocal, tileLength)` → z = x + y = 3.5
- 第 2 次：`Add(zLocal, zLocal, wLocal, tileLength)` → z = z + w = 6.9

在 PyTorch 中可直接写 `x + y + w`，底层 NPU 驱动会自动分解为两次加法指令执行。Ascend C 的 `Add` 接口只支持两个输入，因此三向量相加需要分两步完成。

#### 程序块 5：Golden 计算与验证

```python
golden = torch.full((total_length,), value_x + value_y + value_w, dtype=torch.float32)
return verify_result(z, golden)
```

**说明**：golden 值为 `1.2 + 2.3 + 3.4 = 6.9`。

### 6.3 运行预期结果

```text
计算设备: npu:0
Output: 6.9 6.9 6.9 6.9 6.9 6.9 6.9 6.9 6.9 6.9 6.9 6.9 6.9 6.9 6.9 6.9 6.9 6.9 6.9 6.9...
Golden: 6.9 6.9 6.9 6.9 6.9 6.9 6.9 6.9 6.9 6.9 6.9 6.9 6.9 6.9 6.9 6.9 6.9 6.9 6.9 6.9...
[Success] 精度验证通过！
```

**香橙派实际运行结果截图**：

![阶段三运行结果](images/image3.png)

> **为什么是 6.9？** 三个输入常量之和：1.2 + 2.3 + 3.4 = 6.9。Compute 函数中调用了两次 `Add` 指令：第一次将 x 和 y 相加得到中间结果 3.5，第二次将中间结果与 w 相加得到最终结果 6.9。

---

## 七、阶段四：双向量乘法 —— 计算指令变化

### 7.1 与阶段一的区别

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">项目</th>
<th style="text-align: left;">阶段一</th>
<th style="text-align: left;">阶段四</th>
</tr>
<tr>
<td style="text-align: left;">计算类型</td>
<td style="text-align: left;">加法</td>
<td style="text-align: left;">乘法</td>
</tr>
<tr>
<td style="text-align: left;">Ascend C 接口</td>
<td style="text-align: left;"><code>Add()</code></td>
<td style="text-align: left;"><code>Mul()</code></td>
</tr>
<tr>
<td style="text-align: left;">Python 运算符</td>
<td style="text-align: left;"><code>+</code></td>
<td style="text-align: left;"><code>*</code></td>
</tr>
<tr>
<td style="text-align: left;">Golden 计算</td>
<td style="text-align: left;"><code>value_x + value_y</code></td>
<td style="text-align: left;"><code>value_x * value_y</code></td>
</tr>
<tr>
<td style="text-align: left;">预期输出</td>
<td style="text-align: left;">3.5</td>
<td style="text-align: left;">2.76</td>
</tr>
</table>

### 7.2 代码文件：`code/mul_orangepi.py`

#### 程序块 1：数据参数定义

```python
use_core_num = 8
block_length = 2048
total_length = use_core_num * block_length  # 16384
value_x = 1.2
value_y = 2.3
```

**说明**：数据参数与阶段一完全相同，不修改任何输入值。

#### 程序块 2：NPU 上执行向量乘法（核心变化）

```python
z_npu = x_npu * y_npu       # 关键变化：+ -> *
```

**说明**：对应 Ascend C Compute 函数中将 `Add(zLocal, xLocal, yLocal, tileLength)` 替换为 `Mul(zLocal, xLocal, yLocal, tileLength)`。Python 中将 `+` 运算符替换为 `*` 运算符，底层 NPU 调用乘法向量指令。

#### 程序块 3：Golden 计算与验证（核心变化）

```python
golden = torch.full((total_length,), value_x * value_y, dtype=torch.float32)  # + -> *
return verify_result(z, golden)
```

**说明**：golden 计算从 `value_x + value_y` 改为 `value_x * value_y`，即 `1.2 × 2.3 = 2.76`。验证函数中输出格式改为保留两位小数（`:.2f`）以显示 2.76。

### 7.3 运行预期结果

```text
计算设备: npu:0
Output: 2.76 2.76 2.76 2.76 2.76 2.76 2.76 2.76 2.76 2.76 2.76 2.76 2.76 2.76 2.76 2.76 2.76 2.76 2.76 2.76...
Golden: 2.76 2.76 2.76 2.76 2.76 2.76 2.76 2.76 2.76 2.76 2.76 2.76 2.76 2.76 2.76 2.76 2.76 2.76 2.76 2.76...
[Success] 精度验证通过！
```

**香橙派实际运行结果截图**：

![阶段四运行结果](images/image4.png)

> **与阶段一的关键差异**：阶段一用 `+` 运算符做加法，输出 3.5（1.2+2.3）；阶段四用 `*` 运算符做乘法，输出 2.76（1.2×2.3）。输入数据完全相同，仅计算指令不同，输出结果因此不同。

---

## 八、四阶段结果对比汇总

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">阶段</th>
<th style="text-align: left;">文件</th>
<th style="text-align: left;">输入</th>
<th style="text-align: left;">算子</th>
<th style="text-align: left;">输出</th>
<th style="text-align: left;">核数</th>
</tr>
<tr>
<td style="text-align: left;">阶段一</td>
<td style="text-align: left;"><code>add_8core_orangepi.py</code></td>
<td style="text-align: left;">x=1.2, y=2.3</td>
<td style="text-align: left;">z=x+y</td>
<td style="text-align: left;">3.5</td>
<td style="text-align: left;">8</td>
</tr>
<tr>
<td style="text-align: left;">阶段二</td>
<td style="text-align: left;"><code>add_32core_orangepi.py</code></td>
<td style="text-align: left;">x=2.2, y=2.3</td>
<td style="text-align: left;">z=x+y</td>
<td style="text-align: left;">4.5</td>
<td style="text-align: left;">32</td>
</tr>
<tr>
<td style="text-align: left;">阶段三</td>
<td style="text-align: left;"><code>add3_orangepi.py</code></td>
<td style="text-align: left;">x=1.2, y=2.3, w=3.4</td>
<td style="text-align: left;">z=x+y+w</td>
<td style="text-align: left;">6.9</td>
<td style="text-align: left;">8</td>
</tr>
<tr>
<td style="text-align: left;">阶段四</td>
<td style="text-align: left;"><code>mul_orangepi.py</code></td>
<td style="text-align: left;">x=1.2, y=2.3</td>
<td style="text-align: left;">z=x*y</td>
<td style="text-align: left;">2.76</td>
<td style="text-align: left;">8</td>
</tr>
</table>

### 核心知识点汇总

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">知识点</th>
<th style="text-align: left;">阶段一</th>
<th style="text-align: left;">阶段二</th>
<th style="text-align: left;">阶段三</th>
<th style="text-align: left;">阶段四</th>
</tr>
<tr>
<td style="text-align: left;">NPU 设备初始化</td>
<td style="text-align: left;">✓</td>
<td style="text-align: left;">✓</td>
<td style="text-align: left;">✓</td>
<td style="text-align: left;">✓</td>
</tr>
<tr>
<td style="text-align: left;">Host→Device 搬运</td>
<td style="text-align: left;">✓</td>
<td style="text-align: left;">✓</td>
<td style="text-align: left;">✓</td>
<td style="text-align: left;">✓</td>
</tr>
<tr>
<td style="text-align: left;">Device→Host 搬运</td>
<td style="text-align: left;">✓</td>
<td style="text-align: left;">✓</td>
<td style="text-align: left;">✓</td>
<td style="text-align: left;">✓</td>
</tr>
<tr>
<td style="text-align: left;">精度验证</td>
<td style="text-align: left;">✓</td>
<td style="text-align: left;">✓</td>
<td style="text-align: left;">✓</td>
<td style="text-align: left;">✓</td>
</tr>
<tr>
<td style="text-align: left;">核数配置</td>
<td style="text-align: left;">✓ (8核)</td>
<td style="text-align: left;">✓ (32核)</td>
<td style="text-align: left;">✓ (8核)</td>
<td style="text-align: left;">✓ (8核)</td>
</tr>
<tr>
<td style="text-align: left;">多输入支持</td>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
<td style="text-align: left;">✓</td>
<td style="text-align: left;"></td>
</tr>
<tr>
<td style="text-align: left;">计算指令变化</td>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
<td style="text-align: left;">✓</td>
</tr>
</table>

> **循序渐进的设计**：每个阶段只引入一个变化点，避免学生同时面对多个新概念。通过对比相邻阶段的代码差异，可以清晰地看到"改了什么、为什么改"，降低学习坡度。

---

## 九、香橙派实际操作步骤

以下步骤在**香橙派开发板（昇腾 310B3）**上通过 SSH 执行：

### 步骤一：登录开发板

```bash
# SSH 登录
ssh root@192.168.137.100

# 确认 NPU 芯片型号（应显示昇腾 310B3）
npu-smi info

# 确认 CANN 环境变量
echo $ASCEND_INSTALL_PATH

# 确认 torch_npu 已安装
python3 -c "import torch_npu; print('torch_npu 版本:', torch_npu.__version__)"
```

### 步骤二：进入代码目录

```bash
# 将 code/ 目录上传到开发板后
cd ~/code
ls -la
```

### 步骤三：运行四阶段实验

```bash
# 方式一：逐个运行

# 阶段一：双向量加法（8核）
python3 add_8core_orangepi.py

# 阶段二：双向量加法（32核）
python3 add_32core_orangepi.py

# 阶段三：三向量加法
python3 add3_orangepi.py

# 阶段四：双向量乘法
python3 mul_orangepi.py

# 方式二：一键运行全部
bash run_orangepi.sh all

# 方式三：运行指定阶段
bash run_orangepi.sh 1   # 仅运行阶段一
```

### 步骤四：运行结果验证

成功时终端显示：
```text
计算设备: npu:0
Output: 3.5 3.5 3.5 3.5 3.5 3.5 3.5 3.5 3.5 3.5 3.5 3.5 3.5 3.5 3.5 3.5 3.5 3.5 3.5 3.5...
Golden: 3.5 3.5 3.5 3.5 3.5 3.5 3.5 3.5 3.5 3.5 3.5 3.5 3.5 3.5 3.5 3.5 3.5 3.5 3.5 3.5...
[Success] 精度验证通过！
```

**结果说明**：
- `计算设备: npu:0`：成功识别并使用香橙派 310B3 的 NPU 设备。
- `Output`：NPU 算子实际输出的前 20 个元素。
- `Golden`：CPU 计算的参考值，用于逐元素比对。
- `[Success] 精度验证通过！`：输出与 golden 全部一致，算子实现正确。

> **输入→输出验证**：以阶段一为例，程序生成输入 `x=[1.2, 1.2, ...]` 和 `y=[2.3, 2.3, ...]`（各 16384 个元素），NPU 算子计算得到 `z`，CPU 计算 golden `= [3.5, 3.5, ...]`。验证逻辑检查 `z[i] == golden[i]` 对所有 i 成立，确认算子的输入输出对应关系正确。

**一键运行全部阶段实际结果截图**（`bash run_orangepi.sh`）：

![一键运行全部阶段结果](images/image5.png)

---

## 十、关键问题探究

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">序号</th>
<th style="text-align: left;">问题</th>
</tr>
<tr>
<td style="text-align: left;">1</td>
<td style="text-align: left;">数据为何必须先搬到片上内存（LM）再计算？</td>
</tr>
<tr>
<td style="text-align: left;">2</td>
<td style="text-align: left;">双缓冲真的有用吗？将 BUFFER_NUM 由 2 改为 1，对比性能</td>
</tr>
<tr>
<td style="text-align: left;">3</td>
<td style="text-align: left;">阶段一和阶段二的区别是什么？为什么要做这个扩展？</td>
</tr>
<tr>
<td style="text-align: left;">4</td>
<td style="text-align: left;">阶段三中三向量相加为什么需要两次 Add 指令？</td>
</tr>
<tr>
<td style="text-align: left;">5</td>
<td style="text-align: left;">阶段四中 Add 替换为 Mul，代码需要修改哪些地方？</td>
</tr>
<tr>
<td style="text-align: left;">6</td>
<td style="text-align: left;"><code>x.to(device)</code> 和 <code>z.cpu()</code> 分别对应 Ascend C 中的什么操作？</td>
</tr>
</table>

> 这些问题需在动手验证的基础上作出分析，写入实验报告。

**问题解析提示**：
1. **数据为何先搬到 LM**：AI Core 的 Vector 计算单元只能直接访问片上内存（LM），不能直接从 GM 取数据。在 Python 中对应 `x.to(device)` 将数据从 Host 搬到 NPU Device。
2. **双缓冲的作用**：`BUFFER_NUM=2` 时，当一块数据在 Compute，另一块数据同时在 CopyIn，隐藏了访存延迟。改为 1 后流水线退化为串行，性能下降。
3. **阶段一 vs 阶段二**：核数从 8 增加到 32，数据总量从 16384 增加到 65536，每核处理量保持 2048 不变。目的是理解多核并行与数据切分的关系。
4. **三向量两次 Add**：Ascend C 的 `Add` 接口只支持两个输入，三向量相加需分两步：先 `z = x + y`，再 `z = z + w`。在 Python 中 `x + y + w` 底层也是分两次加法执行。
5. **Add 替换为 Mul**：仅需修改两处——Compute 中的运算符 `+` → `*`，golden 计算中的 `+` → `*`。
6. **to(device) 与 cpu()**：`x.to(device)` 对应 `aclrtMemcpy(ACL_MEMCPY_HOST_TO_DEVICE)`，将数据从 Host 搬到 NPU；`z.cpu()` 对应 `aclrtMemcpy(ACL_MEMCPY_DEVICE_TO_HOST)`，将结果从 NPU 搬回 Host。


---

## 十一、拓展任务（选做）

1. **组合算子**：实现 `z = x * y + w`，融合乘法和加法
2. **新算子尝试**：实现减法（Sub）或除法（Div）算子
3. **性能对比**：对比阶段一（8核）和阶段二（32核）在香橙派上的执行时间
4. **数据类型扩展**：将 float32 类型改为 float16（half），比较精度和性能
5. **自定义计算**：实现 `z = x^2 + y^2`，使用 Mul 和 Add 组合


---

## 课后练习

请根据本节实验内容完成以下题目进行自测。


**第1题**（单选题）Ascend C 算子开发中，数据必须先从哪里搬到哪里再计算？

- A. LM 搬到 GM
- B. GM 搬到 LM
- C. CPU 搬到 NPU
- D. 不需要搬运


In [ ]:
q1 = ''  # 填入你的选项，如 'B'
print(f'第1题答案已记录：{q1}' if q1 else '请填入答案并运行本单元格')

**第2题**（单选题）三级流水线的正确顺序是？

- A. Compute → CopyIn → CopyOut
- B. CopyOut → CopyIn → Compute
- C. CopyIn → Compute → CopyOut
- D. CopyIn → CopyOut → Compute


In [ ]:
q2 = ''  # 填入你的选项，如 'C'
print(f'第2题答案已记录：{q2}' if q2 else '请填入答案并运行本单元格')

**第3题**（单选题）阶段一中使用多少个 AI Core 并行处理？

- A. 4
- B. 8
- C. 16
- D. 32


In [ ]:
q3 = ''  # 填入你的选项，如 'B'
print(f'第3题答案已记录：{q3}' if q3 else '请填入答案并运行本单元格')

**第4题**（单选题）BUFFER_NUM = 2 的作用是什么？

- A. 双缓冲，隐藏访存延迟
- B. 增加计算精度
- C. 减少内存使用
- D. 提高数据精度


In [ ]:
q4 = ''  # 填入你的选项，如 'A'
print(f'第4题答案已记录：{q4}' if q4 else '请填入答案并运行本单元格')

**第5题**（单选题）阶段二相比阶段一的主要变化是？

- A. 核数 8→32
- B. 输入 2→3
- C. Add→Mul
- D. 数据类型变化


In [ ]:
q5 = ''  # 填入你的选项，如 'A'
print(f'第5题答案已记录：{q5}' if q5 else '请填入答案并运行本单元格')

**第6题**（单选题）阶段三相比阶段一的主要变化是？

- A. 核数变化
- B. 输入 2→3，三向量加法
- C. 指令变化
- D. 数据类型变化


In [ ]:
q6 = ''  # 填入你的选项，如 'B'
print(f'第6题答案已记录：{q6}' if q6 else '请填入答案并运行本单元格')

**第7题**（单选题）阶段四将 Add 指令替换为什么？

- A. Sub
- B. Mul
- C. Div
- D. Max


In [ ]:
q7 = ''  # 填入你的选项，如 'B'
print(f'第7题答案已记录：{q7}' if q7 else '请填入答案并运行本单元格')

**第8题**（单选题）Python 代码中 `x.to(device)` 对应 Ascend C 中的什么操作？

- A. aclrtMemcpy(ACL_MEMCPY_HOST_TO_DEVICE)
- B. aclrtMemcpy(ACL_MEMCPY_DEVICE_TO_HOST)
- C. aclInit
- D. aclrtSynchronizeStream


In [ ]:
q8 = ''  # 填入你的选项，如 'A'
print(f'第8题答案已记录：{q8}' if q8 else '请填入答案并运行本单元格')

**第9题**（单选题）阶段三中三向量相加 `z = x + y + w` 底层需要执行几次加法指令？

- A. 1 次
- B. 2 次
- C. 3 次
- D. 4 次


In [ ]:
q9 = ''  # 填入你的选项，如 'B'
print(f'第9题答案已记录：{q9}' if q9 else '请填入答案并运行本单元格')

**第10题**（单选题）阶段四的预期输出值是多少？

- A. 3.5
- B. 4.5
- C. 6.9
- D. 2.76


In [ ]:
q10 = ''  # 填入你的选项，如 'D'
print(f'第10题答案已记录：{q10}' if q10 else '请填入答案并运行本单元格')

**全部作答完成后，运行下方代码查看批改结果：**


In [ ]:
import sys
from pathlib import Path

for candidate in (Path.cwd() / 'answer', Path.cwd().parent / 'answer'):
    if candidate.exists():
        sys.path.insert(0, str(candidate.resolve()))
        break
else:
    raise FileNotFoundError('Cannot find answer directory')
from grade_03 import grade
grade(globals())

## 参考资料

- [昇腾社区 - Ascend C 算子开发](https://hiascend.com/document)
- [CANN 社区样例](https://gitee.com/ascend/samples)
- [torch_npu 使用文档](https://gitee.com/ascend/pytorch)
